# Camada GOLD

**Objetivo desta camada:** obter dados prontos para responder perguntas de negócio.

Modelagem em esquema Estrela.

Tabelas criadas: dimensão - quem, onde, quando; fato - medições.

```
               dim_tempo
                   |
 dim_operadora -- FATO_PRODUCAO -- dim_campo
                   |
              dim_localizacao
```

In [0]:
CATALOGO = "workspace"

## 1. Dimensão CAMPO
Um registro por campo, com sua bacia, ambiente e classificação geológica.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.dim_campo AS
SELECT
  row_number() OVER (ORDER BY campo) AS id_campo,
  campo        AS nome_campo,
  bacia,
  localizacao  AS ambiente, 
  classificacao AS classificacao_geologica 
FROM (
  SELECT DISTINCT campo, bacia, localizacao, classificacao
  FROM {CATALOGO}.silver.producao_poco
  WHERE campo IS NOT NULL
)
""")
display(spark.table(f"{CATALOGO}.gold.dim_campo"))

id_campo,nome_campo,bacia,ambiente,classificacao_geologica
1,ABALONE,CAMPOS,MAR,POS-SAL
2,ACAJÁ-BURIZINHO,RECÔNCAVO,TERRA,TERRA
3,ACAUÃ,POTIGUAR,TERRA,TERRA
4,AGUILHADA,SERGIPE,TERRA,TERRA
5,AGULHA,POTIGUAR,MAR,POS-SAL
6,ALBACORA,CAMPOS,MAR,POS-SAL
7,ALBACORA,CAMPOS,MAR,PRE-SAL
8,ALBACORA LESTE,CAMPOS,MAR,POS-SAL
9,ALBATROZ,ESPÍRITO SANTO,TERRA,TERRA
10,ALTO ALEGRE - COD:4866,POTIGUAR,TERRA,TERRA


## 2. Dimensão OPERADORA

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.dim_operadora AS
SELECT row_number() OVER (ORDER BY operadora) AS id_operadora,
       operadora AS nome_operadora
FROM (SELECT DISTINCT operadora FROM {CATALOGO}.silver.producao_poco
      WHERE operadora IS NOT NULL)
""")
display(spark.table(f"{CATALOGO}.gold.dim_operadora"))

id_operadora,nome_operadora
1,3R BAHIA
2,3R MACAU
3,3R PETROLEUM
4,3R POTIGUAR
5,3R RIO VENTURA
6,ALVOPETRO
7,BARRA BONITA
8,BGM
9,BRASIL REFINARIAS
10,BRAVA ENERGIA S.A.


## 3. Dimensão TEMPO

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.dim_tempo AS
SELECT row_number() OVER (ORDER BY ano, mes) AS id_tempo,
       ano, mes
FROM (SELECT DISTINCT ano, mes FROM {CATALOGO}.silver.producao_poco
      WHERE ano IS NOT NULL AND mes IS NOT NULL)
""")
display(spark.table(f"{CATALOGO}.gold.dim_tempo"))

id_tempo,ano,mes
1,2021,1
2,2021,2
3,2021,3
4,2021,4
5,2021,5
6,2021,6
7,2021,7
8,2021,8
9,2021,9
10,2021,10


## 4. Dimensão LOCALIZAÇÃO

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.dim_localizacao AS
SELECT row_number() OVER (ORDER BY localizacao, campo) AS id_localizacao,
       localizacao,
       campo
FROM (SELECT DISTINCT localizacao, campo FROM {CATALOGO}.silver.producao_poco)
""")
display(spark.table(f"{CATALOGO}.gold.dim_localizacao"))

id_localizacao,localizacao,campo
1,MAR,ABALONE
2,MAR,AGULHA
3,MAR,ALBACORA
4,MAR,ALBACORA LESTE
5,MAR,ANC_ARGONAUTA_ME1
6,MAR,ANC_JUBARTE_NORDESTE
7,MAR,ANC_JUBARTE_SUDOESTE
8,MAR,ANC_MERO
9,MAR,ANC_NORTE_ATAPU
10,MAR,ANC_TUPI


## 5. Tabela FATO
Junta a Silver com as dimensões, trocando os textos pelas chaves (id_*).

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.fato_producao AS
SELECT
  t.id_tempo,
  c.id_campo,
  o.id_operadora,
  loc.id_localizacao,
  s.poco,
  s.petroleo   AS vol_petroleo,
  s.gas        AS vol_gas,
  s.condensado AS vol_condensado,
  s.agua       AS vol_agua
FROM {CATALOGO}.silver.producao_poco s
LEFT JOIN {CATALOGO}.gold.dim_campo      c   ON s.campo     = c.nome_campo
                                              AND s.classificacao = c.classificacao_geologica
LEFT JOIN {CATALOGO}.gold.dim_operadora  o   ON s.operadora = o.nome_operadora
LEFT JOIN {CATALOGO}.gold.dim_tempo      t   ON s.ano = t.ano AND s.mes = t.mes
LEFT JOIN {CATALOGO}.gold.dim_localizacao loc ON s.localizacao = loc.localizacao AND s.campo = loc.campo
""")

qtd = spark.table(f"{CATALOGO}.gold.fato_producao").count()
print(f"Linhas na fato_producao: {qtd:,}")
display(spark.sql(f"SELECT * FROM {CATALOGO}.gold.fato_producao LIMIT 10"))

Linhas na fato_producao: 607,046


id_tempo,id_campo,id_operadora,id_localizacao,poco,vol_petroleo,vol_gas,vol_condensado,vol_agua
58,257,37,292,7-MG-352D-BA,57.9346283216129,4.819903225806452,0.0,2140.023447307742
58,257,37,292,7-MG-402-BA,3.218962440322581,1.2267741935483871,0.0,5.363787650322581
58,257,37,292,7-MG-453D-BA,21.65865638935484,0.05719354838709677,0.0,23.50745473516129
58,257,37,292,7-MGP-72D-BA,0.0,2.71158064516129,0.0068985012903225815,6.837429264193548
58,257,37,292,7-MGP-90D-BA,0.0,9.336677419354839,0.2739110806451613,5.6334378919354835
58,259,4,294,7-MAG-104D-RN,6.645285713548388,0.008258064516129033,0.0,448.42267068354846
58,259,4,294,7-MAG-10-RN,0.0,0.0,0.0,0.0
58,259,4,294,7-MAG-30-RN,4.385620746774193,0.006870967741935484,0.0,82.01486156096773
58,259,4,294,7-MAG-78-RN,0.0,0.0,0.0,0.0
58,259,4,294,7-MAG-93D-RN,0.0,0.0,0.0,0.0
